### Install requiremnts

In [46]:
!pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [47]:
import sys
sys.path.append('../')
import os

from validate_birdset import load_model, to_n_hot, test
import numpy as np
import requests
import zipfile
import glob
import pandas as pd
from datasets import Dataset
from functools import partial
from utils.transform import ValTransformBeans
from utils.event_decoder import EventDecoder
import torch
import json

DEVICE ='cuda'
DOWN_TASKS_ACC = ["walkins", "bats", "cbi", "dogs", "humburgdb", "speech"]
DOWN_TASKS_MAP = ["enabirds", "hiceas", "rfcx", "hainan gibbons"]

# Change this to the directory where the BEANS dataset is stored.
# To download BEANS, see: https://github.com/earthspecies/beans
BEANS_ROOT_PATH = "/mnt/ares_data_2/bellafkir/beans"

#### Download checkpoints

In [48]:
def download_ckpt(url, extract_dir):
   
    zip_filename = "file.zip"
    
    # -----------------------
    # Download ZIP file
    # -----------------------
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(zip_filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    print("Download complete.")

    # -----------------------
    # Extract ZIP file
    # -----------------------
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_filename, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Unzipped to '{extract_dir}'")
    

In [49]:
def evaluate_beans_classification(ckpts, down_task, device='cuda'):
        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        config = configs[0]
        results = dict()
    
        # ---------------------------------------------------------
        # Load dataset metadata and create validation dataframe
        # ---------------------------------------------------------

        # Different datasets have different metadata file structures
        if down_task == "HumBugDB":
            train_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/data/metadata/train.csv")
            val_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/data/metadata/test.csv")
        
        elif down_task == "speech_commands":
            train_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/train.csv")
            val_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/test.csv")
        
        else:
            train_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/annotations.train.csv")
            val_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/annotations.test.csv")

        # ---------------------------------------------------------
        # Prepare label mapping
        # ---------------------------------------------------------
        class_names = sorted(list(set(train_df.label.tolist())))

        # Convert label names into numeric label indices
        val_df['labels'] = val_df['label'].apply(lambda x: class_names.index(x))
        val_df["filepath"] = val_df["path"].apply(lambda x: f"{BEANS_ROOT_PATH}/{x}")

        config.train.label_map = {class_names[i]: i for i in range(len(class_names))}
        config.train.num_classes = len(class_names)

        # Convert pandas dataframe to HuggingFace Dataset
        val_data = Dataset.from_pandas(val_df)
    
        # ---------------------------------------------------------
        # Convert labels to N-hot format
        # ---------------------------------------------------------
        val_data = val_data.map(
        partial(to_n_hot,
                num_classes=config.train.num_classes,
                ),
        batched=True,
        batch_size=300,
        load_from_cache_file=False,
        num_proc=1,
        )

        # ---------------------------------------------------------
        # Define validation transformation pipeline
        # ---------------------------------------------------------
        val_transform = ValTransformBeans(
        config=config,
        train=False,
        event_decoder=EventDecoder(extracted_interval=config.event_decoder.val.extracted_interval,
                                   sample_rate=config.frontend.sample_rate)
        )

        val_data.set_transform(val_transform)

        # ---------------------------------------------------------
        # Create PyTorch DataLoader for validation dataset
        # ---------------------------------------------------------
        val_loader = torch.utils.data.DataLoader(
            val_data,
            num_workers=config.train.num_workers,
            batch_size=config.train.batch_size,
            drop_last=False,
            shuffle=False
        )
    
        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {
                   "top1_acc": []
        }

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            _, _, top1_acc = test(model, val_loader, relevant, center_5s=False, device=device)

            # Store metrics
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

In [50]:
def load_json_dataset(json_path, config, labels, unknown_label=None, window_width=2, window_shift=1):
    # Open the JSON dataset file (each line contains one audio sample description)
    with open(json_path) as f:
        xs = []
        ys = []
        # Iterate through each line (one recording per line)
        for line in f:
            data = json.loads(line)

            # Path to the audio file
            wav_path = data['path']

            # Total duration of the recording
            length = data['length']

            # Compute how many sliding windows can be extracted
            num_windows = int((length - window_width) / window_shift) + 1

            # ---------------------------------------------------------
            # Slide a window across the audio recording
            # ---------------------------------------------------------
            for window_id in range(num_windows):
                # Compute start and end time of the window
                st, ed = window_id * window_shift, window_id * window_shift + window_width
                # Store window boundaries
                offset_st, offset_ed = st, ed
                # Save input sample (audio path + time window)
                xs.append((wav_path, offset_st, offset_ed))
                # Initialize label vector (multi-label classification)
                y = torch.zeros(len(labels))

                # ---------------------------------------------------------
                # Check overlap between window and annotated events
                # --------------------------------------------------------
                for anon in data['annotations']:
                    if anon['label'] not in labels:
                        if unknown_label is not None:
                            # Assign unknown labels to a predefined class
                            label_id = labels.index(unknown_label)
                        else:
                           raise KeyError(f"Unknown label: {anon['label']}")
                    else:
                        # Get class index
                        label_id = labels.index(anon['label'])

                    # ---------------------------------------------------------
                    # Check if annotation overlaps with the current window
                    # ---------------------------------------------------------

                    # Case 1: annotation start or end falls inside the window
                    if (st <= anon['st'] <= ed) or (st <= anon['ed'] <= ed):
                        denom = min(ed - st, anon['ed'] - anon['st'])
                        if denom == 0:
                            continue
                        overlap = (min(ed, anon['ed']) - max(st, anon['st'])) / denom
                        if overlap > .2:
                            y[label_id] = 1

                    # Case 2: annotation fully contains the window
                    if anon['st'] <= st and ed <= anon['ed']:
                        y[label_id] = 1
                # Save label vector for this window
                ys.append(y)
    return xs, ys

In [51]:
def evaluate_beans_detection(ckpts, down_task, device='cuda'):
        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        config = configs[0]
        results = dict()

        # ---------------------------------------------------------
        # Define dataset-specific parameters
        # source: https://github.com/earthspecies/beans/blob/main/datasets.yml
        # ---------------------------------------------------------
        if down_task == "enabirds":
            labels = ["AMCR", "AMGO", "AMRE", "AMRO", "BAWW", "BCCH", "BGGN", "BHCO", "BHVI", "BLJA", "BTNW", "BWWA",
                     "CARW", "COYE", "EATO", "HETH", "HOWA", "KEWA", "LOWA", "NOCA", "NOFL", "OVEN", "RBGR", "RBWO",
                     "RCKI", "REVI", "SCTA", "SWTH", "TUTI", "WBNU", "WITU", "WOTH", "YBCU", "OTHR"]
            unknown_label = "OTHR"
            window_size = 2
            window_shift = 1

        elif down_task == "hiceas":
            labels =  [1]
            unknown_label = None
            window_size = 10 
            window_shift = 5

        elif down_task == "rfcx":
            labels = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
            unknown_label = None
            window_size = 10
            window_shift = 5

        elif down_task == "hainan_gibbons":
            labels= [1, 2, 3]
            unknown_label= 1
            window_size = 4
            window_shift= 2


        config.train.num_classes = len(labels)

        # ---------------------------------------------------------
        # Load test dataset from JSON annotations
        # ---------------------------------------------------------
        val_xs, val_ys = load_json_dataset(f"{BEANS_ROOT_PATH}/data/{down_task}/test.jsonl", config, labels, unknown_label, window_size, window_shift)

        # ---------------------------------------------------------
        #  Convert dataset into a pandas-style dictionary
        # ---------------------------------------------------------
        val_df = {"filepath": [], "labels": [], "start_time": [], "end_time": []}
        for i, item in enumerate(val_xs):
            val_df['filepath'].append(f"{BEANS_ROOT_PATH}/{item[0]}")
            val_df['start_time'].append(item[1])
            val_df['end_time'].append(item[2])
            val_df['labels'].append(val_ys[i].numpy())
            
        # Convert dictionary to HuggingFace Dataset
        val_data = Dataset.from_pandas(pd.DataFrame.from_dict(val_df))

        config.train.label_map = {labels[i]: i for i in range(len(labels))}

        # ---------------------------------------------------------
        # Create validation transform pipeline
        # ---------------------------------------------------------
        val_transform = ValTransformBeans(
        config=config,
        train=False,
        event_decoder=EventDecoder(extracted_interval=config.event_decoder.val.extracted_interval,
                                   sample_rate=config.frontend.sample_rate)
        )

        val_data.set_transform(val_transform)

        # ---------------------------------------------------------
        # Create PyTorch DataLoader
        # ---------------------------------------------------------
        val_loader = torch.utils.data.DataLoader(
            val_data,
            num_workers=config.train.num_workers,
            batch_size=config.train.batch_size,
            drop_last=False,
            shuffle=False
        )
    
        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in labels]

        # Store metrics for each model (for later averaging)
        metrics = {
                   "cmap": [],
                   }

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            _, cmap, _ = test(model, val_loader, relevant, center_5s=False, device=device)

            # Store metrics
            metrics["cmap"].append(cmap)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

In [52]:
# down task: watkins
if not os.path.exists('../ckpts/beans/walkins'):
    download_ckpt('https://next.hessenbox.de/index.php/s/DS6pGXB5oZz5qbY/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/walkins/*')

evaluate_beans_classification(ckpts, "watkins", device=DEVICE)
    

2026-03-15 10:20:57,047 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:20:57,160 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:20:57,179 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 6/6 [00:04<00:00,  1.27it/s]

{'watkins': {'top1_acc': 0.8466076850891113}}


In [53]:
# down task: bats 
if not os.path.exists('../ckpts/beans/bats'):
    download_ckpt('https://next.hessenbox.de/index.php/s/4zkYfT2Aq9YzqWE/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/bats/*')

evaluate_beans_classification(ckpts, "egyptian_fruit_bats", device=DEVICE)
    

2026-03-15 10:21:02,372 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:21:02,523 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:21:02,539 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 32/32 [00:48<00:00,  1.50s/it]

{'egyptian_fruit_bats': {'top1_acc': 0.8025000095367432}}


In [54]:
# down task: cbi 
if not os.path.exists('../ckpts/beans/cbi'):
    download_ckpt('https://next.hessenbox.de/index.php/s/kcidPwoCj4W5GML/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/cbi/*')

evaluate_beans_classification(ckpts, "cbi", device=DEVICE)

2026-03-15 10:21:51,152 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:21:51,267 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:21:51,275 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 57/57 [00:38<00:00,  1.47it/s]


{'cbi': {'top1_acc': 0.8149171471595764}}


In [55]:
# down task: dogs 
if not os.path.exists('../ckpts/beans/dogs'):
    download_ckpt('https://next.hessenbox.de/index.php/s/cLELZ3XeTAR6N4F/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/dogs/*')

evaluate_beans_classification(ckpts, "dogs", device=DEVICE)
    

2026-03-15 10:22:30,903 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:22:31,019 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:22:31,033 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 3/3 [00:06<00:00,  2.20s/it]

{'dogs': {'top1_acc': 0.9784172773361206}}


In [56]:
# down task: humbugdb 
if not os.path.exists('../ckpts/beans/humbugdb'):
    download_ckpt('https://next.hessenbox.de/index.php/s/aEcBPFaXZWm9aMf/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/humbugdb/*')

evaluate_beans_classification(ckpts, "HumBugDB", device=DEVICE)
    

2026-03-15 10:22:38,136 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:22:38,246 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:22:38,260 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 30/30 [00:23<00:00,  1.26it/s]

{'HumBugDB': {'top1_acc': 0.7902097702026367}}


In [57]:
# down task: speech 
if not os.path.exists('../ckpts/beans/speech'):
    download_ckpt('https://next.hessenbox.de/index.php/s/8cJMDnFn3eiGpiP/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/speech/*')

evaluate_beans_classification(ckpts, "speech_commands", device=DEVICE)

2026-03-15 10:23:02,615 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:23:02,725 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:23:02,740 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 172/172 [00:15<00:00, 11.06it/s]

{'speech_commands': {'top1_acc': 0.8526124358177185}}


In [58]:
# down task: enabirds 
if not os.path.exists('../ckpts/beans/enabirds'):
    download_ckpt('https://next.hessenbox.de/index.php/s/52qKwmWpdDjMjtN/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/enabirds/*')

evaluate_beans_detection(ckpts, "enabirds", device=DEVICE)

2026-03-15 10:23:19,010 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:23:19,120 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:23:19,135 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 71/71 [00:10<00:00,  6.70it/s]

{'enabirds': {'cmap': 0.7615470559040949}}


In [59]:
# down task: hiceas 
if not os.path.exists('../ckpts/beans/hiceas'):
    download_ckpt('https://next.hessenbox.de/index.php/s/r3WSdqnwdb7fpRG/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/hiceas/*')

evaluate_beans_detection(ckpts, "hiceas", device=DEVICE)

2026-03-15 10:23:30,341 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:23:30,458 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:23:30,471 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 24/24 [00:17<00:00,  1.40it/s]


{'hiceas': {'cmap': 0.6033437344325436}}


In [60]:
# down task: rfcx 
if not os.path.exists('../ckpts/beans/rfcx'):
    download_ckpt('https://next.hessenbox.de/index.php/s/xMrApXYKCFkGFnB/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/rfcx/*')

evaluate_beans_detection(ckpts, "rfcx", device=DEVICE)

2026-03-15 10:23:48,140 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:23:48,252 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:23:48,267 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 163/163 [01:56<00:00,  1.40it/s]


{'rfcx': {'cmap': 0.20168134723547437}}


In [61]:
# down task: hainan_gibbons 
if not os.path.exists('../ckpts/beans/gibbons'):
    download_ckpt('https://next.hessenbox.de/index.php/s/Zpy7dqEYpmoF6PT/download', extract_dir="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/gibbons/*')

evaluate_beans_detection(ckpts, "hainan_gibbons", device=DEVICE)

2026-03-15 10:25:45,484 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 10:25:45,601 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 10:25:45,613 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 290/290 [00:26<00:00, 10.97it/s]

{'hainan_gibbons': {'cmap': 0.5162510269030967}}
